In [5]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import uproot3 as uproot
import pandas as pd
import numpy as np
import math
from tqdm import tqdm

import kdar_functions as kdar
import general_functions as utils

import importlib



In [6]:
single_run = False

In [7]:
importlib.reload(kdar)
importlib.reload(utils)

<module 'general_functions' from '/Users/bbogart/Documents/analysisCode/kdar_bdt/general_functions.py'>

In [44]:
def print_genie_ratio(var,xmin,xmax,nbins,add_query='',clipped=False):
    query = "is_KDAR==1"+add_query

    clipped_max =  99999999
    clipped_min = -99999999
    if clipped:
        clipped_max = xmax
        clipped_min = xmin  
        
    bins = np.linspace(xmin,xmax,nbins+1)
    
    ubt, _xx = np.histogram(np.clip(kdar_overlay_df.query(query)[var].to_numpy(),clipped_min,clipped_max),
                                    bins=bins,density=False,
                                    weights=np.clip(kdar_overlay_df.query(query)['net_weight'].to_numpy(),0,30))
    genie, _xx = np.histogram(np.clip(kdar_overlay_df.query(query)[var].to_numpy(),clipped_min,clipped_max),
                                    bins=bins,density=False,
                                    weights=np.ones_like(kdar_overlay_df.query(query)['net_weight'].to_numpy()))
    string = ''

    for i in range(len(ubt)):
        if genie[i]!=0:string+=f'{ubt[i]/genie[i]}, '
        else: string+=f'0, '
    print(string)
    
    ubt_sum = np.sum(ubt)
    genie_sum = np.sum(genie)
    
    print(ubt_sum/genie_sum)
    
    string = ''

    for i in range(len(ubt)):
        string+=f'{ubt[i]/ubt_sum}, '
    print(string)

In [10]:
f_kdar_overlay = uproot.open("/Users/bbogart/Documents/data/KDAR_MCC9.10/prodgenie_kdar_overlay_run4b.root")["wcpselection"]
f_kdar_overlay_bdt = f_kdar_overlay["T_BDTvars"].pandas.df(kdar.bdt_vars+kdar.ssm_bdt_vars, flatten=False)
f_kdar_overlay_eval = f_kdar_overlay["T_eval"].pandas.df(kdar.eval_vars+kdar.eval_mc_vars+["weight_spline","weight_cv"], flatten=False)
f_kdar_overlay_pfeval = f_kdar_overlay["T_PFeval"].pandas.df(kdar.pf_eval_vars+kdar.pf_eval_mc_vars, flatten=False)
f_kdar_overlay_kine = f_kdar_overlay["T_KINEvars"].pandas.df(kdar.kine_vars+kdar.kine_mc_vars, flatten=False)
f_kdar_overlay_pot = f_kdar_overlay["T_pot"].pandas.df("pot_tor875good", flatten=False)
kdar_overlay_POT = np.sum(f_kdar_overlay_pot["pot_tor875good"].to_numpy())
kdar_overlay_df = pd.concat([f_kdar_overlay_bdt, f_kdar_overlay_eval, f_kdar_overlay_pfeval, f_kdar_overlay_kine], axis=1, sort=False)

kdar_overlay_df["horncur"] = ["RHC" for i in range(kdar_overlay_df.shape[0])]

del f_kdar_overlay
del f_kdar_overlay_bdt
del f_kdar_overlay_eval
del f_kdar_overlay_pfeval
del f_kdar_overlay_kine

if not single_run: 
    f_kdar_overlay = uproot.open("/Users/bbogart/Documents/data/KDAR_MCC9.10/prodgenie_kdar_overlay_run5.root")["wcpselection"]
    f_kdar_overlay_bdt = f_kdar_overlay["T_BDTvars"].pandas.df(kdar.bdt_vars+kdar.ssm_bdt_vars, flatten=False)
    f_kdar_overlay_eval = f_kdar_overlay["T_eval"].pandas.df(kdar.eval_vars+kdar.eval_mc_vars+["weight_spline","weight_cv"], flatten=False)
    f_kdar_overlay_pfeval = f_kdar_overlay["T_PFeval"].pandas.df(kdar.pf_eval_vars+kdar.pf_eval_mc_vars, flatten=False)
    f_kdar_overlay_kine = f_kdar_overlay["T_KINEvars"].pandas.df(kdar.kine_vars+kdar.kine_mc_vars, flatten=False)
    f_kdar_overlay_pot = f_kdar_overlay["T_pot"].pandas.df("pot_tor875good", flatten=False)
    kdar_overlay_POT = np.sum(f_kdar_overlay_pot["pot_tor875good"].to_numpy())
    kdar_overlay_df_2 = pd.concat([f_kdar_overlay_bdt, f_kdar_overlay_eval, f_kdar_overlay_pfeval, f_kdar_overlay_kine], axis=1, sort=False)

    kdar_overlay_df_2["horncur"] = ["FHC" for i in range(kdar_overlay_df_2.shape[0])]
    
    del f_kdar_overlay
    del f_kdar_overlay_bdt
    del f_kdar_overlay_eval
    del f_kdar_overlay_pfeval
    del f_kdar_overlay_kine

    kdar_overlay_df = pd.concat([kdar_overlay_df, kdar_overlay_df_2], sort=False)

kdar_overlay_df = kdar.apply_goodruns(kdar_overlay_df)

kdar_overlay_df["net_weight"] = kdar_overlay_df["weight_cv"].to_numpy()*kdar_overlay_df["weight_spline"].to_numpy()
print("All events",kdar_overlay_df.shape[0],np.sum(kdar_overlay_df["net_weight"].to_numpy()))

kdar_overlay_df["rse_num"] = (kdar_overlay_df["run"].to_numpy() * 100_000_000_000
                         + kdar_overlay_df["subrun"].to_numpy() * 1_000_000
                         + kdar_overlay_df["event"].to_numpy())
kdar_overlay_df = kdar_overlay_df.drop_duplicates(subset=['rse_num'])
print("Duplicates Dropped",kdar_overlay_df.shape[0],np.sum(kdar_overlay_df["net_weight"].to_numpy()))

kdar_overlay_df = kdar_overlay_df.query("truth_vtxInside==1")
print("In FV",kdar_overlay_df.shape[0],np.sum(kdar_overlay_df["net_weight"].to_numpy()))

kdar_overlay_df["isEXT"] = [0 for i in range(kdar_overlay_df.shape[0])]
kdar_overlay_df["isDirt"] = [0 for i in range(kdar_overlay_df.shape[0])]
kdar_overlay_df["WC_file"] = ["numi_kdar_overlay" for i in range(kdar_overlay_df.shape[0])]
kdar_overlay_df["POTscaled"] = [1 for i in range(kdar_overlay_df.shape[0])]
kdar_overlay_df["is_KDAR"] = [1 for i in range(kdar_overlay_df.shape[0])]

All events 112659 150786.56
Duplicates Dropped 112659 150786.56
In FV 105986 141809.62


In [11]:
kdar_overlay_df = kdar.add_ntrue_nu_angle(kdar_overlay_df)
kdar_overlay_df = kdar.add_truth_muon_info(kdar_overlay_df)
kdar_overlay_df = kdar.add_truth_proton_info(kdar_overlay_df)
kdar_overlay_df = kdar.add_truth_ssm_info(kdar_overlay_df)

100%|████████████████████████████████| 105986/105986 [00:01<00:00, 57547.29it/s]


In [14]:
print_genie_ratio('truth_Emuon',0,165,33,add_query='')

29.775775909423828, 5.27844762802124, 2.2349259853363037, 2.013148307800293, 1.6177287101745605, 1.4681082963943481, 1.3695814609527588, 1.3021619319915771, 1.2746518850326538, 1.2640454769134521, 1.2726532220840454, 1.2881654500961304, 1.302555799484253, 1.2291234731674194, 1.1059412956237793, 0.9767292737960815, 0.8208980560302734, 0.6288830637931824, 0.4538266956806183, 0.20024564862251282, 0.008230452425777912, 0.005226480774581432, 0.0047743055038154125, 0.005445075687021017, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
1.3347268
0.01704932563006878, 0.03660448640584946, 0.0406501404941082, 0.04839957132935524, 0.046680744737386703, 0.051174335181713104, 0.05764422565698624, 0.06265848129987717, 0.07092194259166718, 0.08021452277898788, 0.0893523320555687, 0.10206988453865051, 0.10933329910039902, 0.0796753466129303, 0.048346009105443954, 0.029109444469213486, 0.016724081709980965, 0.00824656244367361, 0.004000511951744556, 0.0010984591208398342, 2.8276073862798512e-05, 1.0603527698549442e-05, 4.8

In [16]:
print_genie_ratio('true_angle_deg',0,180,36,add_query='')

1.1760004758834839, 1.1424545049667358, 1.2302687168121338, 1.170316219329834, 1.1921766996383667, 1.25912344455719, 1.2062466144561768, 1.1948590278625488, 1.1801862716674805, 1.2322030067443848, 1.2102731466293335, 1.2635291814804077, 1.2855082750320435, 1.314375400543213, 1.342113971710205, 1.342334270477295, 1.3677735328674316, 1.3754310607910156, 1.3731329441070557, 1.413373351097107, 1.3374654054641724, 1.3881895542144775, 1.3766905069351196, 1.3632915019989014, 1.382474660873413, 1.366141438484192, 1.3526357412338257, 1.3721020221710205, 1.398517370223999, 1.3653159141540527, 1.3851016759872437, 1.4194411039352417, 1.3639472723007202, 1.395464539527893, 1.3413785696029663, 1.5444661378860474, 
1.3347353
0.0009809477487578988, 0.0032465443946421146, 0.006087716203182936, 0.008148840628564358, 0.010879858396947384, 0.01384060550481081, 0.016704246401786804, 0.01884397864341736, 0.022233309224247932, 0.026540620252490044, 0.028369668871164322, 0.031538378447294235, 0.03478589653968

In [17]:
print_genie_ratio('true_pl',-230,230,46,add_query='')

0, 0, 0, 0, 0.008130860514938831, 0.00988900475203991, 0.01560616958886385, 0.09711748361587524, 0.6748899221420288, 1.0287433862686157, 1.175698161125183, 1.209554672241211, 1.2423216104507446, 1.2520679235458374, 1.2657642364501953, 1.2812749147415161, 1.3172521591186523, 1.349044680595398, 1.4032745361328125, 1.473572015762329, 1.4919062852859497, 1.5120223760604858, 1.6422394514083862, 1.651994228363037, 1.5523507595062256, 1.4154179096221924, 1.3828898668289185, 1.3578362464904785, 1.2685614824295044, 1.2074097394943237, 1.152248740196228, 1.1081222295761108, 1.0636181831359863, 1.0498294830322266, 1.0018165111541748, 0.9510907530784607, 0.8212116956710815, 0.7467365264892578, 0.7255629897117615, 0.8823750019073486, 1.0321691036224365, 0, 0, 0, 0, 0, 
1.3347256
0.0, 0.0, 0.0, 0.0, 1.7243176841930108e-07, 9.087729608836526e-07, 3.0889709705661517e-06, 3.295324131613597e-05, 0.0009016836993396282, 0.004508771933615208, 0.011801675893366337, 0.019383693113923073, 0.02660946547985077,

In [18]:
print_genie_ratio('true_pt',0,230,23,add_query='')

2.144233465194702, 2.4145050048828125, 2.0163393020629883, 1.982542634010315, 1.651333212852478, 1.4916940927505493, 1.447793960571289, 1.3318932056427002, 1.283048152923584, 1.2591665983200073, 1.2445440292358398, 1.2386302947998047, 1.228476643562317, 1.1410045623779297, 0.954211950302124, 0.6240414381027222, 0.0828884020447731, 0.006974637508392334, 0.005599711090326309, 0.005580357275903225, 0, 0, 0, 
1.334726
0.007169561460614204, 0.02502196840941906, 0.03721601888537407, 0.05619870498776436, 0.0654989555478096, 0.07129352539777756, 0.08201919496059418, 0.08767419308423996, 0.09077152609825134, 0.09531274437904358, 0.0996340662240982, 0.09809241443872452, 0.0918346717953682, 0.059098027646541595, 0.025598589330911636, 0.0071905218064785, 0.0003498061851132661, 1.700983557384461e-05, 6.84811584505951e-06, 1.6568022829233087e-06, 0.0, 0.0, 0.0, 


In [25]:
print_genie_ratio('true_Q2',0,200000,40,add_query='')

0.8424771428108215, 0.9564768671989441, 1.06174635887146, 1.1605420112609863, 1.3160988092422485, 1.331663727760315, 1.4723097085952759, 1.5945510864257812, 1.6447150707244873, 1.5257863998413086, 1.4507476091384888, 1.4370760917663574, 1.4227262735366821, 1.356835126876831, 1.3420084714889526, 1.3012001514434814, 1.2852249145507812, 1.2835257053375244, 1.2535635232925415, 1.2332258224487305, 1.2361973524093628, 1.2294436693191528, 1.2168515920639038, 1.2097312211990356, 1.1822973489761353, 1.1475263833999634, 1.020983338356018, 0.9011250138282776, 0.5448446273803711, 0.1692708283662796, 0.015625, 0.013671875, 0.01171875, 0.008413461968302727, 0.0078125, 0.010416666977107525, 0, 0, 0, 0, 
1.3347327
0.0006431897636502981, 0.009912099689245224, 0.021353067830204964, 0.028524812310934067, 0.038507129997015, 0.041974861174821854, 0.04845843464136124, 0.054195113480091095, 0.05783006548881531, 0.05700277164578438, 0.05452752113342285, 0.05672603100538254, 0.057044632732868195, 0.05481513962

In [27]:
print_genie_ratio('true_q',0,500,50,add_query='')

0, 0, 0, 0, 0, 1.0301182270050049, 1.0350590944290161, 0.8690896034240723, 0.7535350322723389, 0.8134915828704834, 0.9335890412330627, 0.9559589624404907, 1.0079591274261475, 1.0440922975540161, 1.0728561878204346, 1.100414514541626, 1.1203749179840088, 1.1695505380630493, 1.2773089408874512, 1.3360916376113892, 1.330297827720642, 1.4477769136428833, 1.5582982301712036, 1.6399139165878296, 1.6034284830093384, 1.4939337968826294, 1.4741125106811523, 1.4333451986312866, 1.374022364616394, 1.3394871950149536, 1.2958873510360718, 1.2630072832107544, 1.2501691579818726, 1.2305328845977783, 1.226580262184143, 1.2013163566589355, 1.1223129034042358, 0.9031451940536499, 0.3494400382041931, 0.015229430049657822, 0.011088709346950054, 0.008413461968302727, 0.0, 0, 0, 0, 0, 0, 0, 0, 
1.3347336
0.0, 0.0, 0.0, 0.0, 0.0, 7.281887519638985e-06, 8.048495510593057e-05, 0.00044233768130652606, 0.0006818209076300263, 0.001615906716324389, 0.003286563092842698, 0.005223669111728668, 0.007317627314478159, 

In [28]:
print_genie_ratio('truth_prim_p_energy',0,145,29,add_query='')

1.2282627820968628, 1.2406466007232666, 1.1596243381500244, 1.1930999755859375, 1.2286535501480103, 1.3208286762237549, 1.3313854932785034, 1.401166319847107, 1.4540162086486816, 1.4499602317810059, 1.46499502658844, 1.4309849739074707, 1.401051640510559, 1.4508578777313232, 1.3783605098724365, 1.384427785873413, 1.3871994018554688, 1.3193479776382446, 1.2982513904571533, 1.832169771194458, 1.5556329488754272, 1.4173704385757446, 1.72066330909729, 1.1200000047683716, 1.157812476158142, 1.015625, 0, 0, 0, 
1.3347281
0.03156126290559769, 0.06035613641142845, 0.0647430419921875, 0.07796423882246017, 0.07564084976911545, 0.07814095914363861, 0.07838904112577438, 0.08425074070692062, 0.07956553250551224, 0.06932954490184784, 0.061794646084308624, 0.05420976132154465, 0.04383532702922821, 0.03859381750226021, 0.030127352103590965, 0.023761708289384842, 0.019367102533578873, 0.01359801646322012, 0.007396947126835585, 0.0036523593589663506, 0.0017264955677092075, 0.0011121543357148767, 0.00059

In [29]:
print_genie_ratio('true_KE',0,200,40,add_query='')

0, 1.1243438720703125, 4.459827899932861, 4.389607906341553, 2.711435317993164, 2.7629950046539307, 2.6279404163360596, 2.3542556762695312, 1.9552340507507324, 1.9312459230422974, 1.775341510772705, 1.71346914768219, 1.60631263256073, 1.505220651626587, 1.4617058038711548, 1.3730310201644897, 1.3189531564712524, 1.2548967599868774, 1.1847941875457764, 1.1577473878860474, 1.2372303009033203, 1.093894362449646, 1.0526732206344604, 1.025895357131958, 1.0078352689743042, 0.9870116710662842, 0.9268350005149841, 0.6982421875, 0.17472957074642181, 0.02690972201526165, 0.004934210330247879, 0.02678571455180645, 0, 0.0, 0, 0, 0, 0, 0, 0, 
1.3347261
0.0, 8.742813224671409e-05, 0.0016078576445579529, 0.00484071671962738, 0.0053668152540922165, 0.009238481521606445, 0.014248535968363285, 0.017491042613983154, 0.02189340442419052, 0.026143617928028107, 0.030960673466324806, 0.037827495485544205, 0.042626895010471344, 0.04819054529070854, 0.05190180987119675, 0.051296137273311615, 0.0502920895814895

In [30]:
print_genie_ratio('true_KE_th35',0,200,40,add_query='')

29.75401496887207, 6.202512741088867, 2.6921567916870117, 2.311936616897583, 1.8304221630096436, 1.6299453973770142, 1.4657069444656372, 1.3773170709609985, 1.3864338397979736, 1.4449567794799805, 1.4233129024505615, 1.4373773336410522, 1.424296498298645, 1.351237416267395, 1.2954790592193604, 1.2222415208816528, 1.1562182903289795, 1.1250417232513428, 1.0879600048065186, 1.1565073728561401, 1.2471122741699219, 1.219164252281189, 1.223528504371643, 1.2259081602096558, 1.2317218780517578, 1.2133146524429321, 1.1248568296432495, 0.9093347787857056, 0.2535211145877838, 0.03356481343507767, 0.0052083334885537624, 0.02566964365541935, 0, 0.015625, 0, 0, 0, 0, 0, 0, 
1.3347247
0.006099628750234842, 0.012539884075522423, 0.01463479083031416, 0.01753619872033596, 0.01685991883277893, 0.019046086817979813, 0.02105380780994892, 0.02398049645125866, 0.02807912603020668, 0.03533175587654114, 0.04264038801193237, 0.05336485430598259, 0.05936325713992119, 0.06922289729118347, 0.05643024295568466, 0.

In [46]:
print_genie_ratio('truth_num_prim_proton',0,4,4,add_query='',clipped=True)

1.3004552125930786, 1.3077735900878906, 1.4460855722427368, 1.3521794080734253, 
1.3347303
0.007207247894257307, 0.6420965790748596, 0.13884036242961884, 0.21185584366321564, 


In [47]:
print_genie_ratio('truth_num_prim_proton_th35',0,4,4,add_query='',clipped=True)

1.244380235671997, 1.4036943912506104, 2.6044065952301025, 18.140625, 
1.3347334
0.46679526567459106, 0.5160296559333801, 0.01679038256406784, 0.0003847073530778289, 


In [50]:
print_genie_ratio('true_angle_P_deg',0,180,36,add_query=' and truth_num_prim_proton>0')

1.354675054550171, 1.2243412733078003, 1.2657428979873657, 1.2860803604125977, 1.3207072019577026, 1.3321197032928467, 1.3114650249481201, 1.345763921737671, 1.3811019659042358, 1.3919199705123901, 1.3965824842453003, 1.4070887565612793, 1.3770499229431152, 1.371178150177002, 1.4179872274398804, 1.4267027378082275, 1.3614535331726074, 1.373749017715454, 1.3080716133117676, 1.3365846872329712, 1.3029346466064453, 1.316831111907959, 1.2773667573928833, 1.2243599891662598, 1.2760014533996582, 1.2229275703430176, 1.173921823501587, 1.2285997867584229, 1.0902513265609741, 1.2158502340316772, 1.1340938806533813, 1.070927381515503, 1.0689483880996704, 0.9863781929016113, 1.3028454780578613, 1.03515625, 
1.3349856
0.008507522754371166, 0.022212699055671692, 0.037059370428323746, 0.04976077750325203, 0.05932892858982086, 0.06538090854883194, 0.06686042994260788, 0.06713336706161499, 0.06546418368816376, 0.06218107417225838, 0.056790824979543686, 0.05290991812944412, 0.04491686075925827, 0.04159

In [51]:
print_genie_ratio('true_angle_P_deg',0,180,36,add_query=' and truth_num_prim_proton_th35>0')

1.3515113592147827, 1.2141399383544922, 1.2525399923324585, 1.2860126495361328, 1.324161410331726, 1.3286248445510864, 1.3073829412460327, 1.3762668371200562, 1.436935544013977, 1.4939184188842773, 1.5301417112350464, 1.5815610885620117, 1.535235047340393, 1.5619398355484009, 1.5742532014846802, 1.6232999563217163, 1.6152396202087402, 1.6263359785079956, 1.6219910383224487, 1.6731996536254883, 1.6033748388290405, 1.6926722526550293, 1.7583775520324707, 1.5914229154586792, 1.572829008102417, 1.9060595035552979, 1.3084310293197632, 1.5895782709121704, 1.371509313583374, 1.8537660837173462, 2.0899832248687744, 1.3758013248443604, 1.6442521810531616, 0.8394097089767456, 3.5843749046325684, 2.8531250953674316, 
1.4253336
0.011485273949801922, 0.030615609139204025, 0.05018220096826553, 0.06606638431549072, 0.07773420959711075, 0.08306916058063507, 0.08071842789649963, 0.0786217600107193, 0.07303869724273682, 0.06842874735593796, 0.06077669933438301, 0.05487231910228729, 0.043413955718278885,

In [33]:
for var in kdar.train_vars_nokineprim:
    print(f'  reader_kdar_lowE.AddVariable("{var}", &tagger.{var});')
    print(f'  reader_kdar_hiE.AddVariable("{var}", &tagger.{var});')

  reader_kdar_lowE.AddVariable("ssm_Nsm", &tagger.ssm_Nsm);
  reader_kdar_hiE.AddVariable("ssm_Nsm", &tagger.ssm_Nsm);
  reader_kdar_lowE.AddVariable("ssm_Nsm_wivtx", &tagger.ssm_Nsm_wivtx);
  reader_kdar_hiE.AddVariable("ssm_Nsm_wivtx", &tagger.ssm_Nsm_wivtx);
  reader_kdar_lowE.AddVariable("ssm_dq_dx_fwd_1", &tagger.ssm_dq_dx_fwd_1);
  reader_kdar_hiE.AddVariable("ssm_dq_dx_fwd_1", &tagger.ssm_dq_dx_fwd_1);
  reader_kdar_lowE.AddVariable("ssm_dq_dx_fwd_2", &tagger.ssm_dq_dx_fwd_2);
  reader_kdar_hiE.AddVariable("ssm_dq_dx_fwd_2", &tagger.ssm_dq_dx_fwd_2);
  reader_kdar_lowE.AddVariable("ssm_dq_dx_fwd_3", &tagger.ssm_dq_dx_fwd_3);
  reader_kdar_hiE.AddVariable("ssm_dq_dx_fwd_3", &tagger.ssm_dq_dx_fwd_3);
  reader_kdar_lowE.AddVariable("ssm_dq_dx_fwd_4", &tagger.ssm_dq_dx_fwd_4);
  reader_kdar_hiE.AddVariable("ssm_dq_dx_fwd_4", &tagger.ssm_dq_dx_fwd_4);
  reader_kdar_lowE.AddVariable("ssm_dq_dx_fwd_5", &tagger.ssm_dq_dx_fwd_5);
  reader_kdar_hiE.AddVariable("ssm_dq_dx_fwd_5", &tagger.s

In [34]:
for var in kdar.train_vars_nokineprim:
    print(f'("{var}","D"),')


("ssm_Nsm","D"),
("ssm_Nsm_wivtx","D"),
("ssm_dq_dx_fwd_1","D"),
("ssm_dq_dx_fwd_2","D"),
("ssm_dq_dx_fwd_3","D"),
("ssm_dq_dx_fwd_4","D"),
("ssm_dq_dx_fwd_5","D"),
("ssm_dq_dx_bck_1","D"),
("ssm_dq_dx_bck_2","D"),
("ssm_dq_dx_bck_3","D"),
("ssm_dq_dx_bck_4","D"),
("ssm_dq_dx_bck_5","D"),
("ssm_d_dq_dx_fwd_12","D"),
("ssm_d_dq_dx_fwd_23","D"),
("ssm_d_dq_dx_fwd_34","D"),
("ssm_d_dq_dx_fwd_45","D"),
("ssm_d_dq_dx_bck_12","D"),
("ssm_d_dq_dx_bck_23","D"),
("ssm_d_dq_dx_bck_34","D"),
("ssm_d_dq_dx_bck_45","D"),
("ssm_max_dq_dx_fwd_3","D"),
("ssm_max_dq_dx_fwd_5","D"),
("ssm_max_dq_dx_bck_3","D"),
("ssm_max_dq_dx_bck_5","D"),
("ssm_max_d_dq_dx_fwd_3","D"),
("ssm_max_d_dq_dx_fwd_5","D"),
("ssm_max_d_dq_dx_bck_3","D"),
("ssm_max_d_dq_dx_bck_5","D"),
("ssm_medium_dq_dx","D"),
("ssm_medium_dq_dx_bp","D"),
("ssm_vtx_activity","D"),
("ssm_pdg","D"),
("ssm_score_mu_fwd","D"),
("ssm_score_p_fwd","D"),
("ssm_score_e_fwd","D"),
("ssm_score_mu_bck","D"),
("ssm_score_p_bck","D"),
("ssm_score_e_bck","D

In [66]:
single_run = False

In [68]:
weight_vars = ["All_UBGenie",'mcweight_filled','AxFFCCQEshape_UBGenie','DecayAngMEC_UBGenie','NormCCCOH_UBGenie','NormNCCOH_UBGenie','RPA_CCQE_UBGenie','ThetaDelta2NRad_UBGenie','Theta_Delta2Npi_UBGenie','VecFFCCQEshape_UBGenie','XSecShape_CCMEC_UBGenie','xsr_scc_Fa3_SCC','xsr_scc_Fv3_SCC','TunedCentralValue_UBGenie']


f_kdar_overlay = uproot.open("/Users/bbogart/Documents/data/KDAR_MCC9.10/sys/prodgenie_kdar_overlay_run4b_UBGenieFluxSmallUni.root")["wcpselection"]
f_kdar_overlay_bdt = f_kdar_overlay["T_BDTvars"].pandas.df(kdar.bdt_vars+kdar.ssm_bdt_vars, flatten=False)
f_kdar_overlay_eval = f_kdar_overlay["T_eval"].pandas.df(kdar.eval_vars+kdar.eval_mc_vars+["weight_spline","weight_cv"], flatten=False)
f_kdar_overlay_pfeval = f_kdar_overlay["T_PFeval"].pandas.df(kdar.pf_eval_vars+kdar.pf_eval_mc_vars, flatten=False)
f_kdar_overlay_kine = f_kdar_overlay["T_KINEvars"].pandas.df(kdar.kine_vars+kdar.kine_mc_vars, flatten=False)
f_kdar_overlay_weight = f_kdar_overlay["T_weight"].pandas.df(weight_vars, flatten=False)
f_kdar_overlay_pot = f_kdar_overlay["T_pot"].pandas.df("pot_tor875good", flatten=False)
kdar_overlay_POT = np.sum(f_kdar_overlay_pot["pot_tor875good"].to_numpy())
kdar_overlay_df = pd.concat([f_kdar_overlay_bdt, f_kdar_overlay_eval, f_kdar_overlay_pfeval, f_kdar_overlay_kine,f_kdar_overlay_weight], axis=1, sort=False)

kdar_overlay_df["horncur"] = ["RHC" for i in range(kdar_overlay_df.shape[0])]

del f_kdar_overlay
del f_kdar_overlay_bdt
del f_kdar_overlay_eval
del f_kdar_overlay_pfeval
del f_kdar_overlay_kine

if not single_run: 
    f_kdar_overlay = uproot.open("/Users/bbogart/Documents/data/KDAR_MCC9.10/sys/prodgenie_kdar_overlay_run5_UBGenieFluxSmallUni.root")["wcpselection"]
    f_kdar_overlay_bdt = f_kdar_overlay["T_BDTvars"].pandas.df(kdar.bdt_vars+kdar.ssm_bdt_vars, flatten=False)
    f_kdar_overlay_eval = f_kdar_overlay["T_eval"].pandas.df(kdar.eval_vars+kdar.eval_mc_vars+["weight_spline","weight_cv"], flatten=False)
    f_kdar_overlay_pfeval = f_kdar_overlay["T_PFeval"].pandas.df(kdar.pf_eval_vars+kdar.pf_eval_mc_vars, flatten=False)
    f_kdar_overlay_kine = f_kdar_overlay["T_KINEvars"].pandas.df(kdar.kine_vars+kdar.kine_mc_vars, flatten=False)
    f_kdar_overlay_weight = f_kdar_overlay["T_weight"].pandas.df(weight_vars, flatten=False)
    f_kdar_overlay_pot = f_kdar_overlay["T_pot"].pandas.df("pot_tor875good", flatten=False)
    kdar_overlay_POT = np.sum(f_kdar_overlay_pot["pot_tor875good"].to_numpy())
    kdar_overlay_df_2 = pd.concat([f_kdar_overlay_bdt, f_kdar_overlay_eval, f_kdar_overlay_pfeval, f_kdar_overlay_kine,f_kdar_overlay_weight], axis=1, sort=False)

    kdar_overlay_df_2["horncur"] = ["FHC" for i in range(kdar_overlay_df_2.shape[0])]
    
    del f_kdar_overlay
    del f_kdar_overlay_bdt
    del f_kdar_overlay_eval
    del f_kdar_overlay_pfeval
    del f_kdar_overlay_kine

    kdar_overlay_df = pd.concat([kdar_overlay_df, kdar_overlay_df_2], sort=False)

kdar_overlay_df = kdar.apply_goodruns(kdar_overlay_df)

kdar_overlay_df["net_weight"] = kdar_overlay_df["weight_cv"].to_numpy()*kdar_overlay_df["weight_spline"].to_numpy()
print("All events",kdar_overlay_df.shape[0],np.sum(kdar_overlay_df["net_weight"].to_numpy()))

kdar_overlay_df["rse_num"] = (kdar_overlay_df["run"].to_numpy() * 100_000_000_000
                         + kdar_overlay_df["subrun"].to_numpy() * 1_000_000
                         + kdar_overlay_df["event"].to_numpy())
kdar_overlay_df = kdar_overlay_df.drop_duplicates(subset=['rse_num'])
print("Duplicates Dropped",kdar_overlay_df.shape[0],np.sum(kdar_overlay_df["net_weight"].to_numpy()))

kdar_overlay_df = kdar_overlay_df.query("truth_vtxInside==1")
print("In FV",kdar_overlay_df.shape[0],np.sum(kdar_overlay_df["net_weight"].to_numpy()))

kdar_overlay_df["isEXT"] = [0 for i in range(kdar_overlay_df.shape[0])]
kdar_overlay_df["isDirt"] = [0 for i in range(kdar_overlay_df.shape[0])]
kdar_overlay_df["WC_file"] = ["numi_kdar_overlay" for i in range(kdar_overlay_df.shape[0])]
kdar_overlay_df["POTscaled"] = [1 for i in range(kdar_overlay_df.shape[0])]
kdar_overlay_df["is_KDAR"] = [1 for i in range(kdar_overlay_df.shape[0])]

All events 105505 141309.62
Duplicates Dropped 105505 141309.62
In FV 105015 140631.31


In [69]:
kdar_overlay_df = kdar.add_ntrue_nu_angle(kdar_overlay_df)
kdar_overlay_df = kdar.add_truth_muon_info(kdar_overlay_df)
kdar_overlay_df = kdar.add_truth_proton_info(kdar_overlay_df)
kdar_overlay_df = kdar.add_truth_ssm_info(kdar_overlay_df)

100%|████████████████████████████████| 105015/105015 [00:01<00:00, 57237.05it/s]


In [70]:

print("All",kdar_overlay_df.shape[0])
kdar_overlay_df = kdar_overlay_df.query('mcweight_filled==1')
print("mcweight_filled==1",kdar_overlay_df.shape[0])

DecayAngMEC_UBGenie = kdar_overlay_df["DecayAngMEC_UBGenie"].to_numpy()
DecayAngMEC_UBGenie_0 = []
DecayAngMEC_UBGenie_1 = []
for i in range(len(DecayAngMEC_UBGenie)):
    DecayAngMEC_UBGenie_0.append( np.clip(np.nan_to_num(DecayAngMEC_UBGenie[i][0],nan=0),0,30))
    DecayAngMEC_UBGenie_1.append( np.clip(np.nan_to_num(DecayAngMEC_UBGenie[i][1],nan=0),0,30))
kdar_overlay_df["DecayAngMEC_UBGenie_0"] = DecayAngMEC_UBGenie_0
kdar_overlay_df["DecayAngMEC_UBGenie_1"] = DecayAngMEC_UBGenie_1

RPA_CCQE_UBGenie = kdar_overlay_df["RPA_CCQE_UBGenie"].to_numpy()
RPA_CCQE_UBGenie_0 = []
RPA_CCQE_UBGenie_1 = []
for i in range(len(RPA_CCQE_UBGenie)):
    RPA_CCQE_UBGenie_0.append( np.clip(np.nan_to_num(RPA_CCQE_UBGenie[i][0],nan=0),0,30))
    RPA_CCQE_UBGenie_1.append( np.clip(np.nan_to_num(RPA_CCQE_UBGenie[i][1],nan=0),0,30))
kdar_overlay_df["RPA_CCQE_UBGenie_0"] = RPA_CCQE_UBGenie_0
kdar_overlay_df["RPA_CCQE_UBGenie_1"] = RPA_CCQE_UBGenie_1


XSecShape_CCMEC_UBGenie = kdar_overlay_df["XSecShape_CCMEC_UBGenie"].to_numpy()
XSecShape_CCMEC_UBGenie_0 = []
XSecShape_CCMEC_UBGenie_1 = []
for i in range(len(XSecShape_CCMEC_UBGenie)):
    XSecShape_CCMEC_UBGenie_0.append( np.clip(np.nan_to_num(XSecShape_CCMEC_UBGenie[i][0],nan=0),0,30))
    XSecShape_CCMEC_UBGenie_1.append( np.clip(np.nan_to_num(XSecShape_CCMEC_UBGenie[i][1],nan=0),0,30))
kdar_overlay_df["XSecShape_CCMEC_UBGenie_0"] = XSecShape_CCMEC_UBGenie_0
kdar_overlay_df["XSecShape_CCMEC_UBGenie_1"] = XSecShape_CCMEC_UBGenie_1

All 105015
mcweight_filled==1 105015


In [82]:
def print_genie_ratio(var,xmin,xmax,nbins,add_query='',clipped=False):
    query = "is_KDAR==1"+add_query

    clipped_max =  99999999
    clipped_min = -99999999
    if clipped:
        clipped_max = xmax
        clipped_min = xmin  
        
    bins = np.linspace(xmin,xmax,nbins+1)
    
    ubt, _xx = np.histogram(np.clip(kdar_overlay_df.query(query)[var].to_numpy(),clipped_min,clipped_max),
                                    bins=bins,density=False,
                                    weights=np.clip(kdar_overlay_df.query(query)['net_weight'].to_numpy(),0,30))
    
    genie, _xx = np.histogram(np.clip(kdar_overlay_df.query(query)[var].to_numpy(),clipped_min,clipped_max),
                                    bins=bins,density=False,
                                    weights=np.ones_like(kdar_overlay_df.query(query)['net_weight'].to_numpy()))
    
    ubt_rpa0, _xx = np.histogram(np.clip(kdar_overlay_df.query(query)[var].to_numpy(),clipped_min,clipped_max),
                                    bins=bins,density=False,
                                    weights=np.clip(kdar_overlay_df.query(query)['weight_spline'].to_numpy(),0,30)*kdar_overlay_df.query(query)['RPA_CCQE_UBGenie_0'].to_numpy())
    
    ubt_rpa1, _xx = np.histogram(np.clip(kdar_overlay_df.query(query)[var].to_numpy(),clipped_min,clipped_max),
                                    bins=bins,density=False,
                                    weights=np.clip(kdar_overlay_df.query(query)['weight_spline'].to_numpy(),0,30)*kdar_overlay_df.query(query)['RPA_CCQE_UBGenie_1'].to_numpy())
    
    ubt_mec, _xx = np.histogram(np.clip(kdar_overlay_df.query(query)[var].to_numpy(),clipped_min,clipped_max),
                                    bins=bins,density=False,
                                    weights=np.clip(kdar_overlay_df.query(query)['weight_spline'].to_numpy(),0,30)*kdar_overlay_df.query(query)['XSecShape_CCMEC_UBGenie_1'].to_numpy())
    
    ubt_sum = np.sum(ubt)
    genie_sum = np.sum(genie)   
    print(ubt_sum/genie_sum)
    ubt_rpa1_sum = np.sum(ubt_rpa1)  
    print(ubt_rpa1_sum/genie_sum)
    ubt_rpa0_sum = np.sum(ubt_rpa0)  
    print(ubt_rpa0_sum/genie_sum)
    ubt_mec_sum = np.sum(ubt_mec)  
    print(ubt_mec_sum/genie_sum)    
    
    string = 'ubt_y = ['
    for i in range(len(ubt)):
        string+=f'{ubt[i]/ubt_sum}, '
    string += ']'
    print(string)
    

    string = 'ubt_rpa0_y = ['
    for i in range(len(ubt_rpa0)):
        string+=f'{ubt_rpa0[i]/ubt_rpa0_sum}, '
    string += ']'
    print(string)


    string = 'ubt_rpa1_y = ['
    for i in range(len(ubt_rpa0)):
        string+=f'{ubt_rpa1[i]/ubt_rpa1_sum}, '
    string += ']'
    print(string)


    string = 'ubt_mec_y = ['
    for i in range(len(ubt_rpa0)):
        string+=f'{ubt_mec[i]/ubt_mec_sum}, '
    string += ']'
    print(string)

In [83]:
print_genie_ratio('truth_Emuon',0,165,33,add_query='')

1.3358487
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.017192518338561058, 0.03673755005002022, 0.04076598212122917, 0.04862692952156067, 0.04661782458424568, 0.05124912038445473, 0.05770522728562355, 0.0626935362815857, 0.07095968723297119, 0.08017011731863022, 0.08933870494365692, 0.10211411118507385, 0.10941926389932632, 0.07947151362895966, 0.04803180694580078, 0.028986701741814613, 0.01666383445262909, 0.008161894045770168, 0.003955142106860876, 0.0010923141380771995, 2.8290793125052005e-05, 1.04698210634524e-05, 4.900767180515686e-06, 2.5617646315367892e-06, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ]
ubt_rpa0_y = [0.012090196018244972, 0.026451272814445766, 0.03150633687318358, 0.03871930075999395, 0.03952917392680369, 0.04572749884259673, 0.05392141811261908, 0.0613146298374196, 0.07217712164172284, 0.08445772545921354, 0.09745243463680585, 0.11559868528354786, 0.12645635566258745, 0.08859655259037016, 0.05016616532177581, 0.02855084387692891, 0.0156309

In [84]:
print_genie_ratio('true_angle_deg',0,180,36,add_query='')

1.3358567
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.0009813609067350626, 0.0031220188830047846, 0.0060196975246071815, 0.008099966682493687, 0.010870606638491154, 0.013772818259894848, 0.01671166718006134, 0.018796948716044426, 0.02227473258972168, 0.026606887578964233, 0.028297364711761475, 0.031521689146757126, 0.0349070280790329, 0.03721030429005623, 0.04092784970998764, 0.04167579486966133, 0.04400710016489029, 0.046312227845191956, 0.04510987550020218, 0.04888163506984711, 0.04581717029213905, 0.04593111202120781, 0.04513106495141983, 0.044427864253520966, 0.04250822588801384, 0.03916831687092781, 0.03692266345024109, 0.03563360124826431, 0.031922630965709686, 0.026828089728951454, 0.023569878190755844, 0.019614202901721, 0.015522642992436886, 0.011576101183891296, 0.006676587741822004, 0.002642277628183365, ]
ubt_rpa0_y = [0.0008121115785491071, 0.0026356167658840805, 0.005181209019599494, 0.007075066033144332, 0.009817902766152275, 0.012491852387410116, 

In [85]:
print_genie_ratio('true_pl',-230,230,46,add_query='')

1.3358468
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.0, 0.0, 0.0, 0.0, 1.738800676776009e-07, 9.164060088551196e-07, 3.114916125923628e-06, 3.3230026019737124e-05, 0.0008920133113861084, 0.004498290363699198, 0.011765033937990665, 0.01935514621436596, 0.02659219689667225, 0.0333409421145916, 0.038553111255168915, 0.044711992144584656, 0.048491157591342926, 0.052529603242874146, 0.05545240268111229, 0.05713684856891632, 0.05748583376407623, 0.05769186094403267, 0.0586509071290493, 0.05929948017001152, 0.05309431999921799, 0.045931506901979446, 0.043510857969522476, 0.040993642061948776, 0.035941723734140396, 0.031172266229987144, 0.027097495272755623, 0.024326330050826073, 0.020314935594797134, 0.01721474900841713, 0.012817864306271076, 0.009439225308597088, 0.005484746303409338, 0.003330744570121169, 0.0019586391281336546, 0.0007616248913109303, 0.00012508113286457956, 0.0, 0.0, 0.0, 0.0, 0.0, ]
ubt_rpa0_y = [0.0, 0.0, 0.0, 0.0, 1.2227651163027042e-07, 6.4443806

In [86]:
print_genie_ratio('true_pt',0,230,23,add_query='')

1.3358476
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.00711398059502244, 0.025051089003682137, 0.0372917577624321, 0.056232206523418427, 0.06559485197067261, 0.0713677629828453, 0.08208444714546204, 0.08773500472307205, 0.09084493666887283, 0.09541373699903488, 0.09951429069042206, 0.09805664420127869, 0.09182436764240265, 0.058862779289484024, 0.025480002164840698, 0.007163368631154299, 0.00034327676985412836, 1.7152700820588507e-05, 6.571489393536467e-06, 1.7820988205130561e-06, 0.0, 0.0, 0.0, ]
ubt_rpa0_y = [0.005874899208463076, 0.0202033164289716, 0.031099847161166358, 0.04707610677977043, 0.05656681217697323, 0.06428356057271853, 0.07644771635203189, 0.08591883164586492, 0.09215907727134323, 0.10077127490769201, 0.10821203470184551, 0.10951873767229732, 0.10424043980794369, 0.06456852114933904, 0.02587318818848271, 0.006858185446463733, 0.00030957451328666265, 1.2011169847811092e-05, 4.6064918803137715e-06, 1.258353617939707e-06, 0.0, 0.0, 0.0, ]
ubt_rpa1_y

In [87]:
print_genie_ratio('true_Q2',0,200000,40,add_query='')

1.3358538
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.0006240743095986545, 0.009875199757516384, 0.021203916519880295, 0.02834193781018257, 0.03853415697813034, 0.042014230042696, 0.048603177070617676, 0.05419665575027466, 0.05810319259762764, 0.05696839094161987, 0.05446906387805939, 0.05679948255419731, 0.05718004330992699, 0.054704438894987106, 0.05320854112505913, 0.050686050206422806, 0.04766880348324776, 0.04479958117008209, 0.042339518666267395, 0.038154054433107376, 0.0338665135204792, 0.029259251430630684, 0.024449946358799934, 0.02035514824092388, 0.015389799140393734, 0.009803279303014278, 0.005238120909780264, 0.002379758981987834, 0.0006695091142319143, 0.0001012450156849809, 6.905600457685068e-06, 3.1186582418740727e-06, 1.5593291209370364e-06, 8.910452038435324e-07, 3.34141958546752e-07, 1.1138065048044155e-07, 0.0, 0.0, 0.0, 0.0, ]
ubt_rpa0_y = [0.0005300968809368371, 0.009634366330822468, 0.02033165109328841, 0.02692959871855019, 0.03582703496720

In [88]:
print_genie_ratio('true_q',0,500,50,add_query='')

1.3358555
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.0, 0.0, 0.0, 0.0, 0.0, 7.343046490859706e-06, 7.333506073337048e-05, 0.0004293608944863081, 0.0006805020966567099, 0.001611596904695034, 0.0032956888899207115, 0.005169576499611139, 0.007335439790040255, 0.009516304358839989, 0.012071049772202969, 0.015145480632781982, 0.018505064770579338, 0.023056643083691597, 0.028905611485242844, 0.03231928497552872, 0.03565021604299545, 0.04080634191632271, 0.049866728484630585, 0.05646657943725586, 0.05801858380436897, 0.059221912175416946, 0.06399592757225037, 0.06424350291490555, 0.06434936821460724, 0.062468208372592926, 0.0600149966776371, 0.05662323534488678, 0.050560515373945236, 0.04337151721119881, 0.034169815480709076, 0.024751534685492516, 0.012926265597343445, 0.003947102930396795, 0.00041355585562996566, 8.576299478590954e-06, 2.450371312079369e-06, 6.682831212856399e-07, 1.1138051547732175e-07, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ]
ubt_rpa0_y = [0.0, 0.0, 0.0

In [89]:
print_genie_ratio('truth_prim_p_energy',0,145,29,add_query='')

1.335849
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.03152713179588318, 0.06041830778121948, 0.06467438489198685, 0.07782603055238724, 0.0756528452038765, 0.0781145915389061, 0.07828238606452942, 0.08417923748493195, 0.07959423214197159, 0.06936878710985184, 0.06173155456781387, 0.05419139191508293, 0.043877895921468735, 0.038700349628925323, 0.030199188739061356, 0.02382596582174301, 0.0194476880133152, 0.013610207475721836, 0.007406171411275864, 0.0036507367622107267, 0.0017256266437470913, 0.001121607143431902, 0.0006009007338434458, 0.00018288768478669226, 8.253335545305163e-05, 7.351149179157801e-06, 0.0, 0.0, 0.0, ]
ubt_rpa0_y = [0.033277942494226324, 0.06314746278741215, 0.06552527099326864, 0.07889612077573359, 0.07521483413452713, 0.07705384394532866, 0.0796522845544634, 0.08856265315954392, 0.08166069030206087, 0.07035789912282135, 0.06159426172011513, 0.05374023589777214, 0.042991184155483766, 0.03665835979925163, 0.028497183382973496, 0.02167644885393

In [90]:
print_genie_ratio('true_KE',0,200,40,add_query='')

1.3358476
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.0, 8.816244371701032e-05, 0.0016213621711358428, 0.004873637109994888, 0.005403030198067427, 0.009299805387854576, 0.014324088580906391, 0.017584940418601036, 0.02196528948843479, 0.0260456595569849, 0.030909722670912743, 0.037936802953481674, 0.04273897409439087, 0.04819861426949501, 0.051902513951063156, 0.051241856068372726, 0.050236109644174576, 0.047093991190195084, 0.043941933661699295, 0.04355193302035332, 0.34114333987236023, 0.03337904438376427, 0.02564351074397564, 0.01988309808075428, 0.014718910679221153, 0.009452029131352901, 0.005195152014493942, 0.001486604567617178, 0.00012953630357515067, 6.905632744746981e-06, 7.796681984473253e-07, 2.5617669052735437e-06, 0.0, 1.1138117628206601e-07, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ]
ubt_rpa0_y = [0.0, 7.464205090575844e-05, 0.0011798912647674983, 0.0036268357776293443, 0.004141632739532369, 0.007178252074664797, 0.011358345717297645, 0.01422838630715171, 0.01

In [91]:
print_genie_ratio('true_KE_th35',0,200,40,add_query='')

1.3358456
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.006150861736387014, 0.012630825862288475, 0.014627484604716301, 0.017666757106781006, 0.01684807613492012, 0.01907990872859955, 0.02109673246741295, 0.02402489073574543, 0.028111688792705536, 0.03520791977643967, 0.042665205895900726, 0.05343062803149223, 0.05942840129137039, 0.06910866498947144, 0.056181080639362335, 0.041052933782339096, 0.0321621410548687, 0.026775626465678215, 0.025269640609622, 0.028579114004969597, 0.26857221126556396, 0.02837701328098774, 0.023438865318894386, 0.018947524949908257, 0.014369416981935501, 0.009392676874995232, 0.005181682296097279, 0.0014839335344731808, 0.0001283112942473963, 6.3487359511782415e-06, 7.796693353157025e-07, 2.561770770626026e-06, 0.0, 1.1138133260146788e-07, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ]
ubt_rpa0_y = [0.0043254256890621085, 0.009079994396955624, 0.011190197518568196, 0.013932008428135684, 0.014037569472984925, 0.016682313049807454, 0.019564611017672994, 

In [92]:
print_genie_ratio('truth_num_prim_proton',0,4,4,add_query='',clipped=True)

1.3358511
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.007185925263911486, 0.6417018175125122, 0.13922609388828278, 0.2118861824274063, ]
ubt_rpa0_y = [0.007589956587266504, 0.6622210958803699, 0.12553332124320501, 0.2046556262891586, ]
ubt_rpa1_y = [0.006191809989665025, 0.5912156153662008, 0.17291099201628476, 0.22968158262784935, ]
ubt_mec_y = [0.00721092883107366, 0.6352763117914909, 0.1392283024191221, 0.2182844569583133, ]


In [93]:
print_genie_ratio('truth_num_prim_proton_th35',0,4,4,add_query='',clipped=True)

1.3358498
0.772105445452197
1.8996055139410744
1.3331230042673534
ubt_y = [0.4664969742298126, 0.5163516998291016, 0.016842586919665337, 0.0003087481018155813, ]
ubt_rpa0_y = [0.4727677596849599, 0.5134457822295682, 0.01356931788144497, 0.00021714020402697297, ]
ubt_rpa1_y = [0.45106234401519035, 0.5235075506114757, 0.024895876879936406, 0.0005342284933974934, ]
ubt_mec_y = [0.5127469739304479, 0.4786243665182194, 0.008605002842224707, 2.3656709108071864e-05, ]


In [95]:
print_genie_ratio('true_angle_P_deg',0,180,36,add_query='')

1.3361118
0.7730295948881007
1.8992035190174401
1.3333499358316523
ubt_y = [0.008519566617906094, 0.02224290370941162, 0.03714607656002045, 0.049794770777225494, 0.05938619747757912, 0.06539639830589294, 0.06672654300928116, 0.06710273772478104, 0.06554291397333145, 0.06220007315278053, 0.05672672018408775, 0.05299930274486542, 0.044844817370176315, 0.041390299797058105, 0.03852122649550438, 0.03511382266879082, 0.030175406485795975, 0.027510570362210274, 0.02316601388156414, 0.021031541749835014, 0.0182733666151762, 0.016813533380627632, 0.013914731331169605, 0.011795236729085445, 0.011772462166845798, 0.010049717500805855, 0.009054843336343765, 0.0077283428981900215, 0.006152339279651642, 0.005626181606203318, 0.00454335194081068, 0.0032674483954906464, 0.0023862191010266542, 0.001636472879908979, 0.0011505906004458666, 0.0002972957445308566, ]
ubt_rpa0_y = [0.007726875185387439, 0.02059572514832578, 0.0345636915142132, 0.04694658614286228, 0.055852015097463445, 0.06217804342279904, 

In [96]:
print_genie_ratio('true_angle_P_deg',0,180,36,add_query=' and truth_num_prim_proton>0')

1.3361126
0.7730295948881007
1.8992035190174401
1.3333499358316523
ubt_y = [0.008519577793776989, 0.022242888808250427, 0.03714597597718239, 0.049794744700193405, 0.05938616394996643, 0.0653965026140213, 0.06672661751508713, 0.06710292398929596, 0.06554292887449265, 0.0622001513838768, 0.05672663077712059, 0.05299927666783333, 0.04484501853585243, 0.041390273720026016, 0.03852120414376259, 0.03511374816298485, 0.030175387859344482, 0.027510611340403557, 0.02316594496369362, 0.021031362935900688, 0.018273357301950455, 0.016813579946756363, 0.01391477882862091, 0.011795230209827423, 0.011772343888878822, 0.010049711912870407, 0.0090550621971488, 0.007728114258497953, 0.006152335554361343, 0.005626402795314789, 0.004543237388134003, 0.0032673345413058996, 0.0023863299284130335, 0.0016363597242161632, 0.0011507021263241768, 0.00029718337464146316, ]
ubt_rpa0_y = [0.007726875185387439, 0.02059572514832578, 0.0345636915142132, 0.04694658614286228, 0.055852015097463445, 0.06217804342279904, 0